# Mini Project 1 — Analysis Notebook (MP1b)

**Your name:** _(fill in)_  
**Dataset:** [ChatGPT reviews (daily updated)](https://www.kaggle.com/datasets/ashishkumarak/chatgpt-reviews-daily-updated) — `ashishkumarak/chatgpt-reviews-daily-updated`.  
**Date:** May 2026  

This notebook extends the Week 5 / A5 pandas exploration with **Section — Analysis**: Plotly visuals tied to MP1 analytical questions, exported as static `.png` files for Canvas / GitHub.

In [ ]:
# Setup — run first (Plotly static export requires kaleido)
import subprocess
import sys


def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])


for pkg in ["pandas", "plotly", "kaleido", "kagglehub"]:
    try:
        __import__(pkg if pkg != "kagglehub" else "kagglehub")
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

print("Setup complete.")

---
## Section 1 — Overview

**Why this dataset:** Public app-store reviews bundle **stated sentiment (stars)**, **text**, and lightweight **social engagement (thumbs-up)** at scale — relevant for HCI / product sensing around generative AI tools.

**Three analytical questions (from MP1a / exploratory plan):**

1. **Polarity skew:** How are star ratings distributed — is sentiment dominated by positive reviews?
2. **Engagement vs. rating:** Do lower-star reviews draw more thumbs-ups on average (amplified complaints), or does engagement track positivity?
3. **Data completeness:** Which columns are unreliable because of systematic missing values (especially version metadata tied to rollout studies)?

**What a practitioner would do with findings:** Product and trust teams can prioritize complaint themes sampling with awareness of imbalance, temper conclusions that depend on exact app/version strings, and interpret community engagement independently from star averages.

---
## Section 2 — Data Profile

**Load reproducibly:** Matches A5 behavior — prefers `kagglehub.dataset_download`, then finds the largest non-demo CSV under that path.

**Profile questions (answered in prose after running the cells):**
- Rows/columns scale (expect ~1M reviews, eight core columns).
- `score`: 1–5 star ratings; `thumbsUpCount`: non‑negative integer engagement; timestamps in `at`; version fields partly missing.
- Structured missingness overlapping `reviewCreatedVersion` and `appVersion`.
- Focus columns: **`score`, `thumbsUpCount`** for Sections 3 charts 1–2; completeness for chart 3.

In [ ]:
from pathlib import Path

import pandas as pd

_EXCLUDE = {"app_info.csv", "app_reviews_demo.csv", "reviews_mini.csv"}


def load_chatgpt_reviews_csv(data_dir: Path) -> tuple[pd.DataFrame, Path]:
    preferred = data_dir / "chatgpt_reviews_kaggle.csv"
    if preferred.exists():
        csv_path = preferred
    else:
        candidates = [
            p
            for p in data_dir.rglob("*.csv")
            if p.name not in _EXCLUDE and not p.name.startswith(".")
        ]
        if not candidates:
            raise FileNotFoundError(
                f"No CSV under {data_dir}. Run `kagglehub.dataset_download(...)`, "
                "or place chatgpt_reviews_kaggle.csv in this notebook folder."
            )
        csv_path = max(candidates, key=lambda p: p.stat().st_size)
    return pd.read_csv(csv_path), csv_path


_root = globals().get("path")

if _root is None:
    try:
        import kagglehub

        path = kagglehub.dataset_download(
            "ashishkumarak/chatgpt-reviews-daily-updated"
        )
    except Exception:
        path = Path(".")

dataset_root = Path(path)

df, _csv_used = load_chatgpt_reviews_csv(dataset_root)
rating_col = "score"

print("Loaded:", _csv_used.resolve())
print("Shape:", df.shape)
df.head(3)

In [ ]:
df.info()
missing_preview = df.isnull().sum()
missing_preview[missing_preview > 0]

---
## Section 3 — Analysis (charts)

Each chart answers **one analytical question**. Images are saved next to this notebook (`A6/`). Bars fit **few discrete rating levels**; a horizontal layout fits **long column names** for missing counts.

### Chart 1 — Distribution of star ratings

**Question:** How concentrated are ratings at each star level?  
**Type:** Vertical bar chart of counts vs. nominal star labels (shown as discrete categories).

In [ ]:
# Save PNGs beside this notebook — open/run from folder `A6` (or cd there first).
_OUT = Path.cwd().resolve()

counts = df[rating_col].value_counts().sort_index()

fig1 = go.Figure(data=[go.Bar(x=counts.index.astype(str), y=counts.values)])
fig1.update_layout(
    title="Distribution of ChatGPT Play Store review star ratings (full dataset)",
    xaxis_title="Star rating (1–5 scale)",
    yaxis_title="Number of reviews (count)",
)

fig1.write_image(_OUT / "chart1_rating_distribution.png")
fig1.show()

**Finding:** Massive imbalance toward 5★ reviews → any subgroup comparison or sampling for themes must normalize or stratify consciously.

### Chart 2 — Mean thumbs-ups by rating

**Question:** Does thumb-based engagement correlate with polarity?  
**Type:** Vertical bar chart of **means** grouped by discrete star rating.

In [ ]:
_OUT = Path.cwd().resolve()

means = df.groupby(rating_col, as_index=False)["thumbsUpCount"].mean()

fig2 = go.Figure(
    data=[go.Bar(x=means[rating_col].astype(str), y=means["thumbsUpCount"])]
)
fig2.update_layout(
    title="Average reader thumbs-ups by star rating (community engagement)",
    xaxis_title="Star rating (1–5 scale)",
    yaxis_title="Mean thumbs-up count (per review)",
)

fig2.write_image(_OUT / "chart2_mean_thumbs_by_rating.png")
fig2.show()

**Finding:** Means sit in a narrow band across stars (**relatively flat**). Interpretation: in this corpus, thumbs-ups are **not** a loud signal separating complaints from praise — qualitative reading still matters.

**Null / dull result clause:** Flat averages are informative: they constrain what engagement alone can diagnose.

### Chart 3 — Missing values by column

**Question:** Which fields are systematically incomplete — especially version metadata affecting temporal or rollout slicing?  
**Type:** Horizontal bar chart (long text labels).

In [ ]:
_OUT = Path.cwd().resolve()

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=True)

fig3 = go.Figure(
    data=[
        go.Bar(
            x=missing.values,
            y=missing.index.astype(str),
            orientation="h",
        )
    ]
)
fig3.update_layout(
    title="Missing values by column (incomplete cells)",
    xaxis_title="Missing cell count",
    yaxis_title="Column name",
)

fig3.write_image(_OUT / "chart3_missing_values_by_column.png")
fig3.show()

**Finding:** **`reviewCreatedVersion`** and **`appVersion`** miss the same counts (aligned missingness pattern) vs. near-complete **`score`** / timestamps — tie version-aware analyses with explicit exclusions or “unknown bucket” labeling.